# Notebook 6 — Interaction Features

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
customers.head(2)

## What Are Interaction Features?

An interaction feature captures the **combined effect** of two (or more) features that
neither feature captures alone. Two customers can have identical `monthly_charges`,
but very different risk if one has 2 months of tenure and the other has 5 years —
the *interaction* of charge level and tenure is what actually matters.

## 1. Feature × Feature (Multiplication)

**When to use:** when the effect of one variable is *amplified or dampened* by another
— classic in pricing, risk, and engagement modeling.

In [ ]:
customers["senior_x_monthly_charges"] = customers["senior_citizen"] * customers["monthly_charges"]
# Interaction: is a senior citizen paying above-average charges? (potential price-sensitivity risk)
customers[["senior_citizen","monthly_charges","senior_x_monthly_charges"]].head()

## 2. Feature + Feature / Feature − Feature

Additive/subtractive interactions are useful when two features represent components of
the same underlying quantity.

In [ ]:
# If we had separate internet + phone charge breakdowns, this would be a true sum.
# Here we demonstrate with an available proxy: contract "commitment score"
contract_score = customers["contract"].map({"Month-to-month":0,"One year":1,"Two year":2})
dependents_score = (customers["dependents"]=="Yes").astype(int)
customers["stability_score"] = contract_score + dependents_score
# Rationale: customers with both a longer contract AND dependents tend to be more "rooted"
customers[["contract","dependents","stability_score"]].drop_duplicates().head()

## 3. Feature / Feature (Ratio Interactions) — the Most Common & Valuable Type

**Real-world example (classic):** `Income / Age` in credit risk models — a young
person with high income signals a very different risk profile than an older person
with the same income.

**This dataset's equivalent:** `monthly_charges / tenure_months` — a new customer
already paying a lot is a different risk profile than a long-tenured customer paying
the same amount.

In [ ]:
customers["charge_per_tenure_month"] = (
    customers["monthly_charges"] / customers["tenure_months"].replace(0, 1)
)
customers[["monthly_charges","tenure_months","charge_per_tenure_month"]].sort_values(
    "charge_per_tenure_month", ascending=False
).head()

## 4. Polynomial Features (Automated Pairwise Interactions)

**When to use:** when you want scikit-learn to systematically generate all pairwise
products/interactions of a numeric feature set — useful for linear models that can't
capture non-linearity on their own, but must be used carefully (see caveat below).

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

num_subset = customers[["monthly_charges", "tenure_months"]].fillna(0)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly_features = poly.fit_transform(num_subset)
poly_names = poly.get_feature_names_out(num_subset.columns)

poly_df = pd.DataFrame(poly_features, columns=poly_names).head()
poly_df

## 5. Domain-Based Interactions

The most valuable interactions are rarely automatic — they come from understanding
*why* two variables would combine meaningfully.

**Example used earlier in Notebook 3:** `contract × payment_method` combo revealed
a churn-risk segment far more concentrated than either variable alone.

## When Interaction Features Help vs When They Create Unnecessary Complexity

**Help when:**
- There's a clear domain hypothesis for *why* two variables interact
- The base features individually have only weak/moderate predictive power
- You're using a linear model that can't discover interactions on its own

**Create unnecessary complexity when:**
- You blindly generate *all* pairwise combinations of many features (`PolynomialFeatures`
  on 30+ columns → hundreds of near-useless, highly correlated features)
- The model is already tree-based (Random Forest / XGBoost/ LightGBM) — these models
  learn feature interactions natively via splits, so many hand-built interactions add
  redundant noise rather than new signal
- Interactions multiply cardinality for categorical crosses, causing sparsity (e.g.
  crossing two 20-category columns creates up to 400 sparse combinations)

> **Senior engineer's rule:** always validate an interaction feature's value in
> Notebook 9/10 (Feature Selection / Importance) rather than assuming it helps just
> because it's plausible.

## Summary — Feature Justification

| Feature | Source | Logic | Reason | Leakage Risk | Decision |
|---|---|---|---|---|---|
| `charge_per_tenure_month` | monthly_charges, tenure_months | ratio | normalizes charge level by relationship length | None | **Retain** |
| `senior_x_monthly_charges` | senior_citizen, monthly_charges | product | tests price sensitivity for senior segment | None | **Needs further analysis** |
| `stability_score` | contract, dependents | additive score | domain-based "rootedness" proxy | None | **Needs further analysis** |
| Auto polynomial features | monthly_charges, tenure_months | sklearn PolynomialFeatures | systematic non-linearity capture | None, but risk of redundancy | **Remove if correlated > 0.9 with existing features (checked in Notebook 9)** |